In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import joblib

# 1. Carregamento dos Dados
df = pd.read_csv('Dengue_diseases_dataset_modified (1).csv')

# OBSERVAÇÃO: O tratamento de outliers foi propositalmente removido para não apagar 
# pacientes com quadros severos de trombocitopenia (queda brusca de plaquetas).

# 2. Separação entre features (X) e target (y)
X = df.drop('dengue_label', axis=1)
y = df['dengue_label']

# 3. Divisão Treino/Teste (80/20 Estratificado)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 4. Pipelines de Pré-processamento
# Variáveis numéricas: Imputação pela mediana (dados ausentes) e Escalonamento (StandardScaler)
# O StandardScaler lidará com a grandeza dos números sem deletar nenhuma linha
numeric_features = ['age', 'hemoglobin_g_dl', 'wbc_count', 'differential_count', 'rbc_count', 'platelet_count', 'platelet_distribution_width']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Variáveis categóricas: Imputação e Transformação (One-Hot Encoding em 'gender')
categorical_features = ['gender']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinando os transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 5. Criação dos Pipelines dos Modelos (Integrando pré-processamento e algoritmo)
knn_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', KNeighborsClassifier(n_neighbors=5))])

svm_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               # probability=True é necessário para plotar a Curva ROC do SVM
                               ('classifier', SVC(kernel='rbf', C=1.0, probability=True, random_state=42))])

# 6. Treinamento
print("Treinando modelos (mantendo todos os casos extremos do dataset)...")
knn_pipeline.fit(X_train, y_train)
svm_pipeline.fit(X_train, y_train)

# 7. Avaliação e Métricas
modelos = {'KNN': knn_pipeline, 'SVM': svm_pipeline}
resultados = {}

# Preparando plotagem para as Matrizes de Confusão
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (nome, modelo) in enumerate(modelos.items()):
    y_pred = modelo.predict(X_test)
    
    # Coleta de Métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    resultados[nome] = [acc, prec, rec, f1]
    
    print(f"\n--- Relatório de Classificação: {nome} ---")
    print(classification_report(y_test, y_pred))
    
    # Plotagem da Matriz de Confusão
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
    disp.plot(ax=axes[idx], cmap='Blues')
    axes[idx].set_title(f'Matriz de Confusão - {nome}')

plt.tight_layout()
plt.show()

# Plotagem da Curva ROC / AUC
plt.figure(figsize=(8, 6))
for nome, modelo in modelos.items():
    y_prob = modelo.predict_proba(X_test)[:, 1] # Pegando probabilidades da classe 1 (Dengue)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{nome} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC - Comparação KNN x SVM')
plt.legend(loc="lower right")
plt.show()

# Comparação gráfica entre os modelos
df_resultados = pd.DataFrame(resultados, index=['Acurácia', 'Precisão', 'Revocação', 'F1-Score'])
df_resultados.plot(kind='bar', figsize=(10, 6), colormap='Set2')
plt.title('Comparação de Métricas entre KNN e SVM')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

# 8. Salvando os modelos para a entrega final
joblib.dump(knn_pipeline, 'modelo_knn.pkl')
joblib.dump(svm_pipeline, 'modelo_svm.pkl')
print("\nModelos salvos com sucesso como 'modelo_knn.pkl' e 'modelo_svm.pkl'!")